# TikTok Shop — Quick Demo (ข้อมูลจำลอง)

**เริ่มใช้งาน:** กด **Runtime → Run all** เพียงครั้งเดียว (และอนุญาต Runtime หาก Colab ถาม) โปรแกรมจะทำทุกขั้นอัตโนมัติ

**เส้นทางข้อมูล:** CSV บน GitHub → SQLite (ER 1) → Parquet Lake → DuckDB (ER 2) → รายงาน KPI

แสดง **Run 1, Run 2, ทดลองรันซ้ำไม่สร้างข้อมูลซ้ำ, Functional/Lineage, Error Recovery 6 กรณี, Performance, กำไร** รวมในหน้าเดียว

> ทุกตัวเลขเป็นข้อมูลจำลอง ไม่มีการเชื่อม TikTok Shop จริง โปรแกรมจะเริ่มข้อมูลทดลองใหม่ใน Colab Runtime ของผู้รันทุกครั้งที่กด Run all โดยไม่แก้ไฟล์ GitHub และไม่ลบไฟล์ใน Google Drive

> ต้องการดูรายละเอียดของแต่ละ Error หรือเปลี่ยนตัวเลือกเพิ่มเติม เปิด [Notebook ฉบับละเอียด](https://colab.research.google.com/github/KankaweeK356/tiktok-shop-pipeline-demo/blob/main/notebooks/TikTok_Shop_Demo.ipynb)


## 00 เตรียมระบบ

**ทำอะไร:** ดึง Code และ Source CSV จาก GitHub พร้อมติดตั้งไลบรารีที่จำเป็น

**เห็นอะไร:** จำนวน Order ตั้งต้น ไม่ต้องกรอก URL หรืออัปโหลดไฟล์

In [ ]:
#@title 00 — เตรียมโปรเจกต์จาก GitHub (อัตโนมัติ)
from pathlib import Path
import os, sys, subprocess, importlib
from IPython.display import display

REPO_URL = "https://github.com/KankaweeK356/tiktok-shop-pipeline-demo"
PROJECT_DIR = Path("/content/TikTok_Shop_Pipeline_Demo")
if (Path.cwd() / "pipeline" / "orchestrator.py").is_file():
    ROOT = Path.cwd()
else:
    ROOT = PROJECT_DIR
    if not ROOT.exists():
        print("กำลังดึง Code และ Source CSV จาก GitHub")
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)])
    else:
        print("ใช้ไฟล์ที่มีอยู่ใน Runtime นี้ (ไม่ git pull ซ้ำ)")

required = ["requirements.txt", "pipeline/orchestrator.py",
            "pipeline/transform.py", "source/initial/Fact_Orders.csv"]
missing = [p for p in required if not (ROOT / p).is_file()]
if missing:
    raise RuntimeError("GitHub/โฟลเดอร์โปรเจกต์มีไฟล์ไม่ครบ: " + ", ".join(missing))

try:
    import duckdb, pyarrow, pandas, matplotlib
except ImportError:
    print("กำลังติดตั้งไลบรารีจาก requirements.txt")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r",
                           str(ROOT / "requirements.txt")])

# Ensure this repository, not a different package named 'pipeline', is imported.
root_path = str(ROOT.resolve())
sys.path = [p for p in sys.path if p != root_path]
sys.path.insert(0, root_path)
for name in list(sys.modules):
    if name == "pipeline" or name.startswith("pipeline."):
        del sys.modules[name]
importlib.invalidate_caches()
from pipeline.orchestrator import run_batch, reset
from pipeline.demo_tests import signature, error_suite, performance_suite
from pipeline.quality import verify, lineage
from pipeline.business import business_report
from pipeline.publication import export_github_reports
import pandas as pd
import matplotlib.pyplot as plt

WORKSPACE = ROOT / "output" / "main"
print("✅ พร้อมใช้งาน | Source: GitHub | Output: Colab Runtime")
print("ข้อมูลตั้งต้น:", len(pd.read_csv(ROOT / "source/initial/Fact_Orders.csv")), "Orders")


## 01 สร้างข้อมูลและตรวจความถูกต้อง

**ทำอะไร:** Initial → Incremental → Replay → Functional Test → Lineage

**เห็นอะไร:** ยอดก่อนและหลัง พร้อมหลักฐานว่า Run 2 ซ้ำไม่ทำให้ข้อมูลซ้ำ

*ขั้นนี้จะล้างเฉพาะพื้นที่ทดลอง `output/main` ใน Colab แล้วเริ่มใหม่เสมอ เพื่อให้กด Run all ซ้ำได้*

In [ ]:
#@title 01 — Run 1 + Run 2 + ตรวจข้อมูลซ้ำ (อัตโนมัติ)
# Always reset only temporary Colab demo outputs to make 'Run all' repeatable.
reset(WORKSPACE)
initial = run_batch(WORKSPACE, "initial")
before = signature(WORKSPACE)
incremental = run_batch(WORKSPACE, "incremental")
after = signature(WORKSPACE)
replay = run_batch(WORKSPACE, "incremental")
replay_state = signature(WORKSPACE)
assert before["orders"] < after["orders"], "ข้อมูล Incremental ไม่เพิ่มรายการที่จ่ายเงิน"
assert replay_state == after, "รัน Incremental ซ้ำแล้วข้อมูลเปลี่ยน"
assert initial["functional_fails"] == incremental["functional_fails"] == 0

validated = verify(WORKSPACE / "er1.sqlite", WORKSPACE / "er2.duckdb",
                   WORKSPACE / "reports", "final")
traced = lineage(WORKSPACE / "er1.sqlite", WORKSPACE / "er2.duckdb", "final",
                 WORKSPACE / "lake" / "control", WORKSPACE / "reports", limit=6)
assert validated["failed"] == 0 and traced["traced_items"] > 0

display(pd.DataFrame([
    {"การทดลอง":"Run 1 — Initial", "รายการขายที่จ่ายแล้ว":before["orders"],
     "ยอดขายสุทธิ (บาท)":before["sales"]},
    {"การทดลอง":"Run 2 — Incremental", "รายการขายที่จ่ายแล้ว":after["orders"],
     "ยอดขายสุทธิ (บาท)":after["sales"]},
    {"การทดลอง":"Run 2 ซ้ำ", "รายการขายที่จ่ายแล้ว":replay_state["orders"],
     "ยอดขายสุทธิ (บาท)":replay_state["sales"]}
]))
print("✅ ตรวจยอด:", validated["passed"], "/", validated["checks"], "ข้อ")
print("✅ Idempotency: รัน Run 2 ซ้ำแล้วรายการและยอดขายไม่เพิ่ม")
print("✅ Lineage: ตามข้อมูลได้", traced["traced_items"], "รายการ")
print("ไฟล์ฐานข้อมูลและ Lake อยู่ใน", WORKSPACE)


## 02 ทดลอง Error และแก้กลับ

**ทำอะไร:** จำลองความผิดพลาด 6 กรณี แยกพื้นที่จากฐานข้อมูลหลัก แล้ว Retry อัตโนมัติ

**เห็นอะไร:** ตรวจพบอะไร, เริ่มใหม่จากขั้นไหน และผ่านหรือไม่

In [ ]:
#@title 02 — Error Injection → Recovery (ครบ 6 กรณี)
# Uses independent workspaces: main sales databases are not modified.
errors = error_suite(ROOT / "output" / "test", cases=None)
view = errors[["case","stage","detected","retry_from","passed"]].copy()
view.rename(columns={"case":"กรณี", "stage":"จุดที่ผิด",
                     "detected":"ตรวจพบ Error", "retry_from":"กลับไปรันจาก",
                     "passed":"กู้คืนสำเร็จ"}, inplace=True)
display(view)
assert len(errors) == 6 and errors["passed"].all(), "มี Error Case ที่ยังไม่ผ่าน"
print("✅ Error Recovery ผ่าน", int(errors["passed"].sum()), "/6 กรณี")
print("การทดสอบนี้ไม่กระทบข้อมูลยอดขายหลัก")


## 03 วัดประสิทธิภาพและดูผลธุรกิจ

**ทำอะไร:** วัดเวลาประมวลผล 2 ขนาด, สรุป KPI และเตรียมรายงาน

**เห็นอะไร:** Latency / Throughput / ยอดขาย / กำไร / ZIP รายงานจริง

In [ ]:
#@title 03 — Performance + ยอดขาย/กำไร + รายงานผล
perf = performance_suite(ROOT / "output" / "performance", sizes=(100, 300))
monthly, product_groups = business_report(WORKSPACE / "er2.duckdb",
                                          WORKSPACE / "reports")
assert not monthly.empty and not product_groups.empty
print("Performance (ผลจริงของเครื่องที่รัน):")
display(perf[["initial_source_orders","latency_seconds",
              "throughput_paid_items_per_second"]].rename(columns={
    "initial_source_orders":"จำนวน Orders",
    "latency_seconds":"เวลา (วินาที)",
    "throughput_paid_items_per_second":"รายการต่อวินาที"}))

print("ยอดขายและกำไร (ข้อมูลจำลอง):")
display(monthly[["Year_Month","Net_Revenue","Contribution_Profit",
                 "Profit_Protection_Status"]].rename(columns={
    "Year_Month":"เดือน","Net_Revenue":"ยอดขายสุทธิ (บาท)",
    "Contribution_Profit":"กำไรหลังต้นทุน (บาท)",
    "Profit_Protection_Status":"ผลเกณฑ์กำไร"}))
fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.bar(monthly["Year_Month"], monthly["Net_Revenue"], label="Net revenue")
ax.plot(monthly["Year_Month"], monthly["Contribution_Profit"], marker="o", label="Profit")
ax.set(title="Revenue and profit — mock data", ylabel="Baht", xlabel="Month")
ax.legend()
plt.tight_layout()
plt.show()

published = export_github_reports(ROOT)
print("✅ รายงานครบแล้ว:", published["zip_path"])
print("เปิดดูไฟล์ใน Colab: Files → TikTok_Shop_Pipeline_Demo → output")


## 04 Google Drive — ทางเลือก

โดยปกติ **ข้ามได้เลย** ไม่ต้องให้สิทธิ์ Drive ผลลัพธ์ทั้งหมดแสดงใน Colab และเก็บใน Runtime อยู่แล้ว

หากต้องการเก็บถาวร ให้ติ๊ก Save แล้วรัน Cell นี้เพียง Cell เดียว ส่วนการล้าง Snapshot ให้ใช้ [Notebook ฉบับละเอียด](https://colab.research.google.com/github/KankaweeK356/tiktok-shop-pipeline-demo/blob/main/notebooks/TikTok_Shop_Demo.ipynb) หัวข้อ 05.4

In [ ]:
#@title 04 — บันทึกผลลง Google Drive (ไม่บังคับ)
SAVE_TO_DRIVE = False #@param {type:"boolean"}
if SAVE_TO_DRIVE:
    from google.colab import drive
    from pipeline.drive_storage import save_to_drive
    drive.mount("/content/drive")
    saved = save_to_drive(ROOT, Path("/content/drive/MyDrive"))
    print("✅ บันทึก Snapshot ใน Google Drive:", saved["path"])
else:
    print("ไม่ได้เชื่อม Drive — ผลการทดลองยังอยู่ใน Colab Runtime")
    print("หากต้องการบันทึกภายหลัง ให้ติ๊ก SAVE_TO_DRIVE แล้วรันเฉพาะ Cell นี้")
